# 04 · Exploratory turnaround propagation pressure

**RQ2.** How much of the delay observed in a given hour is inherited from earlier
legs of the same aircraft?

BTS reports `LateAircraftDelay` as a cause, which is a label rather than a
mechanism. Reconstructing the aircraft rotation turns it into a quantity: the
previous leg's delay minus the scheduled turnaround buffer estimates pressure under a minimum-turn assumption.
This is not a causal allocation of realised arrival delay. Whole aircraft histories are sampled and matched in UTC.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

In [ ]:
from team01 import features, models, viz

from team01.rotation_pressure import analyse_file
cascade = analyse_file(config.flights_file(), tails=256)

print(f"rotations                    {cascade['rotations']:,}")
print(f"share of flights             {cascade['rotation_share_of_flights']:.1%}")
print(f"pressure / arrival ratio     {cascade['pressure_to_positive_arrival_ratio']:.3f}")

## 4.1 The arithmetic, on one aircraft

A rotation counts only when the aircraft continues from where it landed and the
scheduled ground time is plausible (15 minutes to 24 hours). The inherited minutes
are `previous delay − (scheduled turn − 25-minute minimum turn)`.

In [ ]:
pd.DataFrame(cascade["sensitivity"]).T

## 4.2 Where it concentrates

Inspect the supported hourly pattern and 15/25/45-minute turnaround sensitivity.
Do not infer a monotone daily pattern or a causal attribution from this estimate.

In [ ]:
cascade["hourly"][["dep_hour", "flights", "mean_inherited", "inherited_share"]].tail(10)

In [ ]:
viz.fig_cascade_hour(cascade["hourly"])